In [1]:
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd

from nibabel.processing import resample_to_output

from code_execution.functions.find_z import find_z_argmax
from resources import find_tilt_error, get_template_images
from resources.cut_off_edges import cut_off_edges
from resources.find_pitch_error_y_y import find_pitch_error
from resources.find_rotation_error import find_rotation_error_by_profile
from resources.find_tilt_error import find_tilt_error_by_profile
from resources.rotate_image import rotate_volume
from resources.standarize_image import standardize_image

In [2]:
project_dir = Path.cwd().parents[1]

# Jeżeli notebook znajduje się np. w katalogu src:
if not (project_dir / "data").exists():
    project_dir = project_dir.parent

niftis_dir = project_dir / "data" / "niftis"
labels_path = niftis_dir / "train_labels_JNDlMjr.csv"

outputs_dir = project_dir / "src" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

labels_df = pd.read_csv(labels_path)

target_column = "is_pathologic"

labels = labels_df.set_index("uid")[target_column]

nifti_paths = sorted(niftis_dir.glob("*.nii.gz"))

print("Katalog projektu:", project_dir)
print("Liczba obrazów:", len(nifti_paths))
print("Liczba etykiet:", len(labels))

Katalog projektu: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons
Liczba obrazów: 1363
Liczba etykiet: 1362


In [4]:
from resources.preprocess_one_nifti import preprocess_one_nifti
from resources.find_rotation_error import find_rotation_error, find_rotation_error_by_similarity
from notebooks.rotate_nifti_by_errors import rotate_nifti_by_errors
from resources.find_tilt_error import find_tilt_error, find_tilt_error_by_similarity
from resources.find_pitch_error_y_y import find_pitch_error_by_similarity
from notebooks.register_datscan_ants import register_to_template
from resources.find_z import calculate_z_profile, find_z_area_center
from resources.find_y import calculate_y_profile, find_y_area_center
from resources.find_x import calculate_x_profile, find_x_area_center
from resources.get_template_images import get_template_images
volumes = []
y_labels = []
uids = []
template_images = get_template_images()
processed_images_dir = (
    project_dir
    / "data"
)

processed_images_dir.mkdir(
    parents=True,
    exist_ok=True,
)

for index, nifti_path in enumerate(nifti_paths, start=1):
    uid = nifti_path.name.removesuffix(".nii.gz")

    if uid not in labels.index:
        print(f"Brak etykiety dla {uid}")
        continue
    image = nib.load(nifti_path)

    image_for_errors = preprocess_one_nifti(
    image
    )

    # ==================================================
    # Rotation wokół Z
    # ==================================================

    rotation_errors = [
        find_rotation_error_by_similarity(
            image_for_errors,
            template_image,
        )
        for template_image in template_images
        ]

    median_rotation_error = int(
        np.median(rotation_errors)
    )


    # ==================================================
    # Tilt wokół Y
    # ==================================================

    tilt_errors = [
        find_tilt_error_by_similarity(
            image_for_errors,
            template_image,
        )
        for template_image in template_images
    ]

    median_tilt_error = int(
        np.median(tilt_errors)
    )


    # ==================================================
    # Pitch wokół X
    # ==================================================

    pitch_errors = [
        find_pitch_error_by_similarity(
            image_for_errors,
            template_image,
        )
        for template_image in template_images
    ]

    median_pitch_error = int(
        np.median(pitch_errors)
    )


    print(
        f"rotation={rotation_errors} "
        f"-> {median_rotation_error}",
        flush=True,
    )

    print(
        f"tilt={tilt_errors} "
        f"-> {median_tilt_error}",
        flush=True,
    )

    print(
        f"pitch={pitch_errors} "
        f"-> {median_pitch_error}",
        flush=True,
    )
    print(f"{index}/{len(nifti_paths)}: {uid}")

    processed_image = cut_off_edges(image)

    processed_image = cut_off_edges(
        processed_image,
        x_left=35,
        x_right=35,
        y_front=40,
        y_back=40,
        z_down=30,
        z_up=30,
    )

    processed_image = standardize_image(
        processed_image,
        percent_top=0.02,
    )

    processed_image = resample_to_output(
        processed_image,
        voxel_sizes=(2.46, 2.46, 2.46),
        order=1,
    )
    processed_volume = processed_image.get_fdata(
    dtype=np.float32,
    )
    x_profile = calculate_x_profile(processed_volume)
    x_centre = find_x_area_center(x_profile)
    y_profile = calculate_y_profile(processed_volume)
    y_center = find_y_area_center(y_profile)
    z_profile = calculate_z_profile(processed_volume)
    z_center = find_z_argmax(z_profile)
    xyz_center = [x_centre, y_center, z_center]
    print('mediany =',median_rotation_error, median_tilt_error, median_pitch_error)
    processed_image = rotate_volume(processed_image , center=xyz_center, angle=median_rotation_error, axis='z')
    processed_image = rotate_volume(processed_image , center=xyz_center, angle=median_tilt_error, axis='y')
    processed_image = rotate_volume(processed_image , center=xyz_center, angle=median_pitch_error, axis='x')

    processed_image = cut_off_edges(
        processed_image,
        x_left=30,
        x_right=30,
        y_front=30,
        y_back=30,
        z_down=20,
        z_up=25,
    )

    processed_image = cut_off_edges(
        processed_image,
        x_left=25,
        x_right=25,
        y_front=30,
        y_back=30,
        z_down=10,
        z_up=20,
    )

    # Zapis przetworzonego obrazu do data.
    processed_image_path = (
        processed_images_dir
        / f"__{uid}.nii.gz"
    )

    nib.save(
        processed_image,
        str(processed_image_path),
    )

    print(
        f"Zapisano: {processed_image_path.name}",
        flush=True,
    )


    volume = processed_image.get_fdata(dtype=np.float32)

    volumes.append(volume)
    y_labels.append(int(labels.loc[uid]))
    uids.append(uid)
    print('image przeprocesowany -', nifti_path.name)

KeyboardInterrupt: 

In [3]:
from collections import Counter

shape_counts = Counter(volume.shape for volume in volumes)
print(shape_counts)
X = np.stack(volumes).astype(np.float32)
y = np.asarray(y_labels, dtype=np.int64)
uids = np.asarray(uids)

print("X:", X.shape)
print("y:", y.shape)
X = X[:, np.newaxis, ...]

print(X.shape)

NameError: name 'volumes' is not defined

In [ ]:
np.savez_compressed(
    project_dir / "data" / "processed_volumes_with_rotation_by_similarity.npz",
    uid=uids,
    X=X.astype(np.float16),
    y=y,
)

In [4]:
data = np.load(
    project_dir / "data" / "processed_volumes_with_rotation_by_similarity.npz",
    allow_pickle=False,
)

X = data["X"].astype(np.float32)
y = data["y"].astype(np.int64)
uids = data["uid"]

In [5]:
from torch.utils.data import Dataset
import torch


class VolumeDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        volume = torch.from_numpy(self.X[index]).float()

        if volume.ndim == 3:
            volume = volume.unsqueeze(0)

        label = torch.tensor(self.y[index], dtype=torch.float32)

        return volume, label

In [6]:
import torch.nn as nn


class Simple3DCNNLarge(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(
                in_channels=1,
                out_channels=8,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(
                in_channels=8,
                out_channels=32,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),

            nn.AdaptiveAvgPool3d(1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x.squeeze(1)

In [7]:
from sklearn.model_selection import train_test_split

indices = np.arange(len(y))

train_val_indices, test_indices = train_test_split(
    indices,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

X_train_val = X[train_val_indices]
y_train_val = y[train_val_indices]
uids_train_val = uids[train_val_indices]

X_test = X[test_indices]
y_test = y[test_indices]
uids_test = uids[test_indices]

print("X_train_val:", X_train_val.shape)
print("X_test:", X_test.shape)
print("y_train_val:", np.unique(y_train_val, return_counts=True))
print("y_test:", np.unique(y_test, return_counts=True))

X_train_val: (1089, 1, 50, 60, 30)
X_test: (273, 1, 50, 60, 30)
y_train_val: (array([0, 1]), array([492, 597]))
y_test: (array([0, 1]), array([123, 150]))


In [8]:
import copy
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from datetime import datetime

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Urządzenie:", device)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

epochs = 1000
batch_size = 8
early_stopping_patience = 50
min_delta = 0.0001

fold_results = []
fold_model_paths = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_val, y_train_val), start=1):
    print(f"\n===== FOLD {fold} =====")

    X_train = X_train_val[train_idx]
    X_val = X_train_val[val_idx]

    y_train = y_train_val[train_idx]
    y_val = y_train_val[val_idx]

    train_dataset = VolumeDataset(X_train, y_train)
    val_dataset = VolumeDataset(X_val, y_val)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )

    validation_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    model = Simple3DCNNLarge().to(device)
    # model = SmallResNet3D(dropout=0.35).to(device)
    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.0001,
        weight_decay=0.0001,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience=15,
    min_lr=1e-4,
    )

    best_validation_loss = float("inf")
    best_model_state = None
    epochs_without_improvement = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for volumes_batch, labels_batch in train_loader:
            volumes_batch = volumes_batch.to(device)
            labels_batch = labels_batch.to(device)

            optimizer.zero_grad()

            logits = model(volumes_batch)
            loss = criterion(logits, labels_batch)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(labels_batch)

        train_loss /= len(train_dataset)

        model.eval()
        validation_loss = 0.0

        with torch.no_grad():
            for volumes_batch, labels_batch in validation_loader:
                volumes_batch = volumes_batch.to(device)
                labels_batch = labels_batch.to(device)

                logits = model(volumes_batch)
                loss = criterion(logits, labels_batch)

                validation_loss += loss.item() * len(labels_batch)

        validation_loss /= len(val_dataset)
        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(validation_loss)
        new_lr = optimizer.param_groups[0]["lr"]
        if new_lr < current_lr:
            print(
                f"LR zmniejszony: {current_lr:.8f} -> {new_lr:.8f}. "
                f"Reset no_improve.",
                flush=True,
            )
            epochs_without_improvement = 0


        if validation_loss < best_validation_loss - min_delta:
            best_validation_loss = validation_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(
            f"Fold {fold} | Epoka {epoch + 1}/{epochs} | "
            f"train: {train_loss:.4f} | "
            f"validation: {validation_loss:.4f} | "
            f"lr: {current_lr:.8f} | "
            f"no_improve: {epochs_without_improvement}"
            )

        if epochs_without_improvement > early_stopping_patience:
            print(
                f"Early stopping: fold {fold}, epoka {epoch + 1}, "
                f"best validation loss: {best_validation_loss:.4f}"
            )
            break

    model.load_state_dict(best_model_state)

    model_path = f"model_{run_id}_fold_{fold}_{best_validation_loss}.pth"
    torch.save(model.state_dict(), model_path)

    fold_model_paths.append(model_path)
    fold_results.append(best_validation_loss)

    print(f"Fold {fold} najlepszy validation loss: {best_validation_loss:.4f}")
    print(f"Zapisano: {model_path}")

print("\nWyniki foldów:", fold_results)
print("Średni validation loss:", np.mean(fold_results))

Urządzenie: cuda

===== FOLD 1 =====
Fold 1 | Epoka 1/1000 | train: 0.6894 | validation: 0.6893 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 2/1000 | train: 0.6887 | validation: 0.6885 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 3/1000 | train: 0.6882 | validation: 0.6878 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 4/1000 | train: 0.6874 | validation: 0.6865 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 5/1000 | train: 0.6838 | validation: 0.6806 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 6/1000 | train: 0.6796 | validation: 0.6724 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 7/1000 | train: 0.6727 | validation: 0.6672 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 8/1000 | train: 0.6612 | validation: 0.6540 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 9/1000 | train: 0.6548 | validation: 0.6352 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 10/1000 | train: 0.6303 | validation: 0.6214 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 11/1000 | train: 0.6208 | v

In [9]:
from pathlib import Path

OUTPUTS_DIR = Path("../../outputs").resolve()

fold_model_paths = [
    OUTPUTS_DIR / f"2model_fold_{fold}.pth"
    for fold in range(1, 6)
]

print("Katalog roboczy:", Path.cwd())
print("Katalog modeli:", OUTPUTS_DIR)

for model_path in fold_model_paths:
    print(
        model_path.name,
        "istnieje:",
        model_path.exists(),
    )

Katalog roboczy: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons\src\notebooks\EDA6
Katalog modeli: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons\src\outputs
2model_fold_1.pth istnieje: True
2model_fold_2.pth istnieje: True
2model_fold_3.pth istnieje: True
2model_fold_4.pth istnieje: True
2model_fold_5.pth istnieje: True


In [10]:
state_dict = torch.load(
    fold_model_paths[0],
    map_location="cpu",
    weights_only=True,
)

for key in [
    "classifier.2.weight",
    "classifier.2.bias",
    "classifier.5.weight",
    "classifier.5.bias",
    "classifier.7.weight",
    "classifier.7.bias",
]:
    value = state_dict.get(key)

    print(
        key,
        tuple(value.shape) if value is not None else "BRAK",
    )

classifier.2.weight (32, 64)
classifier.2.bias (32,)
classifier.5.weight (16, 32)
classifier.5.bias (16,)
classifier.7.weight (1, 16)
classifier.7.bias (1,)


In [11]:
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score, f1_score
import numpy as np
import torch
from torch.utils.data import DataLoader

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
test_dataset = VolumeDataset(X_test, y_test)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

all_fold_probs = []

for model_path in fold_model_paths:
    model = Simple3DCNNLarge().to(device)
    model.load_state_dict(
        torch.load(model_path, map_location=device,weights_only=True,)
    )
    model.eval()

    fold_probs = []

    with torch.no_grad():
        for volumes_batch, labels_batch in test_loader:
            volumes_batch = volumes_batch.to(device)

            logits = model(volumes_batch)
            probs = torch.sigmoid(logits)

            fold_probs.extend(
                probs.cpu().numpy()
            )

    fold_probs = np.array(fold_probs)
    all_fold_probs.append(fold_probs)

all_fold_probs = np.array(all_fold_probs)

# średnia predykcji z 5 modeli
test_probs = all_fold_probs.mean(axis=0)

# zabezpieczenie pod log loss
test_probs = np.clip(test_probs, 0.001, 0.999)

test_pred_class = (test_probs >= 0.5).astype(int)

print("Liczba modeli:", len(fold_model_paths))
print("Shape all_fold_probs:", all_fold_probs.shape)
print("Min pred:", test_probs.min())
print("Max pred:", test_probs.max())

print("Log loss:", log_loss(y_test, test_probs))
print("AUROC:", roc_auc_score(y_test, test_probs))
print("Accuracy:", accuracy_score(y_test, test_pred_class))
print("F1:", f1_score(y_test, test_pred_class))

Liczba modeli: 5
Shape all_fold_probs: (5, 273)
Min pred: 0.044420507
Max pred: 0.999
Log loss: 0.32010313868522644
AUROC: 0.938590785907859
Accuracy: 0.8681318681318682
F1: 0.8807947019867549


In [12]:
train_dataset = VolumeDataset(
    X_train_val,
    y_train_val,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

all_fold_train_probs = []

for model_path in fold_model_paths:
    print("Wczytuję:", model_path.name)

    model = Simple3DCNNLarge().to(device)

    state_dict = torch.load(
        model_path,
        map_location=device,
        weights_only=True,
    )

    model.load_state_dict(
        state_dict,
        strict=True,
    )

    model.eval()

    fold_train_probs = []

    with torch.inference_mode():
        for volumes_batch, _ in train_loader:
            volumes_batch = volumes_batch.to(device)

            logits = model(volumes_batch)
            probabilities = torch.sigmoid(logits)

            fold_train_probs.extend(
                probabilities.cpu().numpy()
            )

    all_fold_train_probs.append(
        np.asarray(
            fold_train_probs,
            dtype=np.float32,
        )
    )

Wczytuję: 2model_fold_1.pth
Wczytuję: 2model_fold_2.pth
Wczytuję: 2model_fold_3.pth
Wczytuję: 2model_fold_4.pth
Wczytuję: 2model_fold_5.pth


In [13]:
all_fold_train_probs = np.asarray(
    all_fold_train_probs,
    dtype=np.float32,
)

print(
    "Shape all_fold_train_probs:",
    all_fold_train_probs.shape,
)

assert all_fold_train_probs.shape == (
    5,
    len(y_train_val),
)

meta_train_eda6 = pd.DataFrame({
    "uid": np.asarray(uids_train_val),
    "y_true": np.asarray(y_train_val),
})

for fold_index in range(5):
    meta_train_eda6[
        f"p_eda6_fold_{fold_index + 1}"
    ] = np.clip(
        all_fold_train_probs[fold_index],
        0.001,
        0.999,
    )

meta_train_path = (
    Path.cwd()
    / "meta_train_predictions_eda6.csv"
)

meta_train_eda6.to_csv(
    meta_train_path,
    index=False,
)

print("Zapisano:", meta_train_path)
print("Shape:", meta_train_eda6.shape)

display(meta_train_eda6.head())

Shape all_fold_train_probs: (5, 1089)
Zapisano: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons\src\notebooks\EDA6\meta_train_predictions_eda6.csv
Shape: (1089, 7)


,uid,y_true,p_eda6_fold_1,p_eda6_fold_2,p_eda6_fold_3,p_eda6_fold_4,p_eda6_fold_5
0,xjn6fokz,0,0.153959,0.285420,0.213447,0.333463,0.266485
1,po6j0m3l,1,0.945385,0.831022,0.973613,0.934644,0.973652
2,45bg34iq,0,0.509882,0.632679,0.220572,0.405176,0.359730
3,g5rifwr0,1,0.997282,0.994022,0.994718,0.978875,0.982774
4,6e07fpy1,0,0.103359,0.166019,0.097250,0.091975,0.107103


In [14]:
class Simple3DCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool3d(1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x.squeeze(1)

In [15]:
from pathlib import Path

ants_model_paths = [
    Path.cwd() / f"ants_aug_model_fold_{fold}.pth"
    for fold in range(1, 6)
]

for model_path in ants_model_paths:
    if not model_path.exists():
        raise FileNotFoundError(model_path)

    print(model_path.name)

ants_aug_model_fold_1.pth
ants_aug_model_fold_2.pth
ants_aug_model_fold_3.pth
ants_aug_model_fold_4.pth
ants_aug_model_fold_5.pth


In [16]:
ants_train_dataset = VolumeDataset(
    X_train_val,
    y_train_val,
)

ants_train_loader = DataLoader(
    ants_train_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

In [17]:
all_ants_train_probs = []

for model_path in ants_model_paths:
    print("Wczytuję:", model_path.name)

    model = Simple3DCNN().to(device)

    state_dict = torch.load(
        model_path,
        map_location=device,
        weights_only=True,
    )

    model.load_state_dict(
        state_dict,
        strict=True,
    )

    model.eval()

    fold_probs = []

    with torch.inference_mode():
        for volumes_batch, _ in ants_train_loader:
            volumes_batch = volumes_batch.to(device)

            logits = model(volumes_batch)
            probabilities = torch.sigmoid(logits)

            fold_probs.extend(
                probabilities.cpu().numpy()
            )

    all_ants_train_probs.append(
        np.asarray(
            fold_probs,
            dtype=np.float32,
        )
    )

all_ants_train_probs = np.asarray(
    all_ants_train_probs,
    dtype=np.float32,
)

print(
    "Shape:",
    all_ants_train_probs.shape,
)

assert all_ants_train_probs.shape == (
    5,
    len(y_train_val),
)

Wczytuję: ants_aug_model_fold_1.pth
Wczytuję: ants_aug_model_fold_2.pth
Wczytuję: ants_aug_model_fold_3.pth
Wczytuję: ants_aug_model_fold_4.pth
Wczytuję: ants_aug_model_fold_5.pth
Shape: (5, 1089)


In [18]:
meta_train_antspyx = pd.DataFrame({
    "uid": np.asarray(uids_train_val),
    "y_true": np.asarray(y_train_val),
})

for fold_index in range(5):
    meta_train_antspyx[
        f"p_antspyx_fold_{fold_index + 1}"
    ] = np.clip(
        all_ants_train_probs[fold_index],
        0.001,
        0.999,
    )

meta_train_antspyx.to_csv(
    Path.cwd()
    / "meta_train_predictions_antspyx.csv",
    index=False,
)

display(meta_train_antspyx.head())

,uid,y_true,p_antspyx_fold_1,p_antspyx_fold_2,p_antspyx_fold_3,p_antspyx_fold_4,p_antspyx_fold_5
0,xjn6fokz,0,0.206324,0.044938,0.593849,0.170358,0.631159
1,po6j0m3l,1,0.594830,0.494980,0.708146,0.860590,0.795568
2,45bg34iq,0,0.457912,0.233320,0.771167,0.474506,0.576462
3,g5rifwr0,1,0.576289,0.739176,0.907781,0.928097,0.962708
4,6e07fpy1,0,0.061857,0.006627,0.183021,0.014706,0.087207


In [19]:
assert np.array_equal(
    meta_train_antspyx["uid"].astype(str),
    meta_train_eda6["uid"].astype(str),
)

assert np.array_equal(
    meta_train_antspyx["y_true"],
    meta_train_eda6["y_true"],
)

print("UID i etykiety są identyczne.")

UID i etykiety są identyczne.


In [20]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss,
    roc_auc_score,
)

y_true = (
    meta_train_antspyx["y_true"]
    .to_numpy()
    .astype(int)
)

prediction_columns = [
    f"p_antspyx_fold_{fold}"
    for fold in range(1, 6)
]

for column in prediction_columns:
    probabilities = (
        meta_train_antspyx[column]
        .to_numpy()
    )

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    print(f"\n{column}")

    print(
        "Log loss:",
        log_loss(y_true, probabilities),
    )

    print(
        "AUROC:",
        roc_auc_score(y_true, probabilities),
    )

    print(
        "Accuracy:",
        accuracy_score(y_true, predictions),
    )

    print(
        "F1:",
        f1_score(y_true, predictions),
    )


p_antspyx_fold_1
Log loss: 0.36311575770378113
AUROC: 0.928568996745244
Accuracy: 0.8402203856749312
F1: 0.8443649373881932

p_antspyx_fold_2
Log loss: 0.44738954305648804
AUROC: 0.9405121815037245
Accuracy: 0.8126721763085399
F1: 0.7980198019801981

p_antspyx_fold_3
Log loss: 0.3510257303714752
AUROC: 0.93422736991189
Accuracy: 0.8576675849403123
F1: 0.8740861088545898

p_antspyx_fold_4
Log loss: 0.3260006010532379
AUROC: 0.9359773120344268
Accuracy: 0.8576675849403123
F1: 0.8639157155399473

p_antspyx_fold_5
Log loss: 0.34077736735343933
AUROC: 0.9373970121610764
Accuracy: 0.8549127640036731
F1: 0.8746031746031746


In [21]:
class ProfileMLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_size, 1024),
            nn.LayerNorm(1024),
            nn.Mish(),
            nn.Dropout(0.3),

            nn.Linear(1024, 128),
            nn.LayerNorm(128),
            nn.Mish(),
            nn.Dropout(0.3),

            nn.Linear(128, 32),
            nn.LayerNorm(32),
            nn.Mish(),
            nn.Dropout(0.2),

            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.network(x).squeeze(1)

In [22]:
from pathlib import Path
import torch

PROFILE_MODEL_PATH = (
    Path.cwd()
    / "profile_model.pth"
)
state_dict = torch.load(
    PROFILE_MODEL_PATH,
    map_location=device,
    weights_only=True,
)
profile_model = ProfileMLP(
    input_size=state_dict["network.0.weight"].shape[1],
).to(device)

profile_model.load_state_dict(
    torch.load(
        PROFILE_MODEL_PATH,
        map_location=device,
        weights_only=True,
    ),
    strict=True,
)

profile_model.eval()

print("Wczytano:", PROFILE_MODEL_PATH)

Wczytano: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons\src\notebooks\EDA6\profile_model.pth


In [23]:
data = np.load(
    project_dir / "data" / "processed_profiles_by_similarity.npz",
    allow_pickle=False,
)
print(data.files)
uids = data["uid"]
y = data["is_pathologic"]

x_profile=data['x_profile']
y_profile=data['y_profile']
z_profile=data['z_profile']
rotation_x_profile=data['rotation_x_profile']
tilt_x_profiles=data['tilt_x_profiles']
pitch_y_profiles=data['pitch_y_profiles']

X = np.concatenate(
    [
        x_profile,
        y_profile,
        z_profile,
        rotation_x_profile,
        tilt_x_profiles,
        pitch_y_profiles,
    ],
    axis=1,
)

['uid', 'is_pathologic', 'x_profile', 'y_profile', 'z_profile', 'rotation_x_profile', 'tilt_x_profiles', 'pitch_y_profiles']


In [24]:
reference_train = pd.read_csv(
    Path.cwd()
    / "meta_train_predictions_eda6.csv"
)


In [25]:
uids_as_str = np.asarray(
    uids
).astype(str)

reference_train["uid"] = (
    reference_train["uid"].astype(str)
)

if len(np.unique(uids_as_str)) != len(uids_as_str):
    raise ValueError(
        "UID w pliku profili nie są unikalne."
    )

In [26]:
X_profile_train = X[
    train_val_indices
].astype(np.float32)

X_profile_test = X[
    test_indices
].astype(np.float32)

y_profile_train = np.asarray(
    y[train_val_indices]
).astype(int)

y_profile_test = np.asarray(
    y[test_indices]
).astype(int)

uids_profile_train = np.asarray(
    uids[train_val_indices]
).astype(str)

uids_profile_test = np.asarray(
    uids[test_indices]
).astype(str)

In [27]:
print(
    "Profile train:",
    X_profile_train.shape,
)

print(
    "Profile test:",
    X_profile_test.shape,
)

assert X_profile_train.shape[1] == 1217
assert X_profile_test.shape[1] == 1217

assert np.array_equal(
    y_profile_train,
    np.asarray(y_train_val).astype(int),
)

assert np.array_equal(
    y_profile_test,
    np.asarray(y_test).astype(int),
)

Profile train: (1089, 1217)
Profile test: (273, 1217)


In [28]:
X_profile_train_tensor = torch.from_numpy(
    X_profile_train
).float()

X_profile_test_tensor = torch.from_numpy(
    X_profile_test
).float()

X_profiles_reduced = np.concatenate(
    [
        x_profile,
        y_profile,
        z_profile,
        rotation_x_profile,
    ],
    axis=1,
).astype(np.float32)
X_profile_train = (
    X_profiles_reduced[
        train_val_indices
    ]
)

X_profile_test = (
    X_profiles_reduced[
        test_indices
    ]
)

print(
    "Train:",
    X_profile_train.shape,
)

print(
    "Test:",
    X_profile_test.shape,
)

assert X_profile_train.shape[1] == 495
assert X_profile_test.shape[1] == 495
X_profile_train_tensor = torch.as_tensor(
    X_profile_train,
    dtype=torch.float32,
)

X_profile_test_tensor = torch.as_tensor(
    X_profile_test,
    dtype=torch.float32,
)

Train: (1089, 495)
Test: (273, 495)


In [29]:
profile_model.eval()

with torch.inference_mode():
    profile_train_probs = torch.sigmoid(
        profile_model(
            X_profile_train_tensor.to(device)
        )
    ).cpu().numpy().reshape(-1)

    profile_test_probs = torch.sigmoid(
        profile_model(
            X_profile_test_tensor.to(device)
        )
    ).cpu().numpy().reshape(-1)

profile_train_probs = np.clip(
    profile_train_probs,
    0.001,
    0.999,
)

profile_test_probs = np.clip(
    profile_test_probs,
    0.001,
    0.999,
)

In [30]:
meta_train_profiles = pd.DataFrame({
    "uid": uids_profile_train,
    "y_true": y_profile_train,
    "p_profiles": profile_train_probs,
})

meta_train_profiles.to_csv(
    Path.cwd()
    / "meta_train_predictions_profiles.csv",
    index=False,
)

test

In [31]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    roc_auc_score,
)


MODEL_PATH = (
    Path.cwd()
    / "2final_meta_classifier.joblib"
)

saved_meta = joblib.load(
    MODEL_PATH
)

meta_model = saved_meta["model"]
feature_columns = saved_meta["feature_columns"]

print("Cechy modelu:")

for column in feature_columns:
    print(column)

Cechy modelu:
p_eda6_fold_1
p_eda6_fold_2
p_eda6_fold_3
p_eda6_fold_4
p_eda6_fold_5
p_antspyx_fold_1
p_antspyx_fold_2
p_antspyx_fold_3
p_antspyx_fold_4
p_antspyx_fold_5
p_profiles
p_ants_native_mean


In [38]:
ants_similarity_test_dataset = VolumeDataset(
    X_test,
    y_test,
)

ants_similarity_test_loader = DataLoader(
    ants_similarity_test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

all_ants_similarity_test_probs = []

for model_path in ants_model_paths:
    print("Wczytuję:", model_path.name)

    model = Simple3DCNN().to(device)

    state_dict = torch.load(
        model_path,
        map_location=device,
        weights_only=True,
    )

    model.load_state_dict(
        state_dict,
        strict=True,
    )

    model.eval()

    fold_test_probs = []

    with torch.inference_mode():
        for volumes_batch, _ in ants_similarity_test_loader:
            volumes_batch = volumes_batch.to(device)

            logits = model(volumes_batch)
            probabilities = torch.sigmoid(logits)

            fold_test_probs.extend(
                probabilities.cpu().numpy()
            )

    all_ants_similarity_test_probs.append(
        np.asarray(
            fold_test_probs,
            dtype=np.float32,
        )
    )

all_ants_similarity_test_probs = np.asarray(
    all_ants_similarity_test_probs,
    dtype=np.float32,
)

print(
    "Shape:",
    all_ants_similarity_test_probs.shape,
)

Wczytuję: ants_aug_model_fold_1.pth
Wczytuję: ants_aug_model_fold_2.pth
Wczytuję: ants_aug_model_fold_3.pth
Wczytuję: ants_aug_model_fold_4.pth
Wczytuję: ants_aug_model_fold_5.pth
Shape: (5, 273)


In [39]:
n_test = len(y_test)

all_fold_probs = np.asarray(
    all_fold_probs
)

all_ants_similarity_test_probs = np.asarray(
    all_ants_similarity_test_probs
)

profile_test_probs = np.asarray(
    profile_test_probs
).reshape(-1)

assert all_fold_probs.shape == (5, n_test)
assert all_ants_similarity_test_probs.shape == (5, n_test)
assert profile_test_probs.shape == (n_test,)
assert len(uids_test) == n_test

print("Liczba przypadków testowych:", n_test)

Liczba przypadków testowych: 273


In [40]:
meta_test = pd.DataFrame({
    "uid": np.asarray(uids_test).astype(str),
    "y_true": np.asarray(y_test).astype(int),
})

for fold_index in range(5):
    meta_test[
        f"p_eda6_fold_{fold_index + 1}"
    ] = np.clip(
        all_fold_probs[fold_index],
        0.001,
        0.999,
    )

for fold_index in range(5):
    meta_test[
        f"p_antspyx_fold_{fold_index + 1}"
    ] = np.clip(
        all_ants_similarity_test_probs[fold_index],
        0.001,
        0.999,
    )

meta_test["p_profiles"] = np.clip(
    profile_test_probs,
    0.001,
    0.999,
)

In [41]:
ants_native = pd.read_csv(
    Path.cwd()
    / "meta_ants_predictions_from_report.csv"
)

ants_native["uid"] = (
    ants_native["uid"].astype(str)
)

ants_native = ants_native.rename(
    columns={
        "is_pathologic": "p_ants_native_mean"
    }
)

meta_test = meta_test.merge(
    ants_native[
        ["uid", "p_ants_native_mean"]
    ],
    on="uid",
    how="left",
    validate="one_to_one",
)

if meta_test["p_ants_native_mean"].isna().any():
    missing = meta_test.loc[
        meta_test["p_ants_native_mean"].isna(),
        "uid",
    ]

    raise ValueError(
        f"Brakuje natywnej predykcji ANTs "
        f"dla {len(missing)} UID."
    )

In [42]:
missing_features = [
    column
    for column in feature_columns
    if column not in meta_test.columns
]

if missing_features:
    raise ValueError(
        f"Brakuje cech: {missing_features}"
    )

X_meta_test = meta_test[
    feature_columns
].to_numpy(dtype=np.float32)

y_meta_test = meta_test[
    "y_true"
].to_numpy(dtype=int)

print("X_meta_test:", X_meta_test.shape)

X_meta_test: (273, 12)


In [43]:
final_test_probs = meta_model.predict_proba(
    X_meta_test
)[:, 1]

final_test_probs = np.clip(
    final_test_probs,
    0.001,
    0.999,
)

final_test_classes = (
    final_test_probs >= 0.5
).astype(int)

print(
    "TEST log loss:",
    log_loss(
        y_meta_test,
        final_test_probs,
    ),
)

print(
    "TEST AUROC:",
    roc_auc_score(
        y_meta_test,
        final_test_probs,
    ),
)

print(
    "TEST accuracy:",
    accuracy_score(
        y_meta_test,
        final_test_classes,
    ),
)

print(
    "TEST F1:",
    f1_score(
        y_meta_test,
        final_test_classes,
    ),
)

print("\nMacierz omyłek:")

print(
    confusion_matrix(
        y_meta_test,
        final_test_classes,
    )
)

print("\nRaport:")

print(
    classification_report(
        y_meta_test,
        final_test_classes,
    )
)

TEST log loss: 0.33367079496383667
TEST AUROC: 0.9434146341463415
TEST accuracy: 0.8644688644688645
TEST F1: 0.8794788273615635

Macierz omyłek:
[[101  22]
 [ 15 135]]

Raport:
              precision    recall  f1-score   support

           0       0.87      0.82      0.85       123
           1       0.86      0.90      0.88       150

    accuracy                           0.86       273
   macro avg       0.87      0.86      0.86       273
weighted avg       0.86      0.86      0.86       273



In [44]:
eda6_mean = np.mean(
    all_fold_probs,
    axis=0,
)

ants_similarity_mean = np.mean(
    all_ants_similarity_test_probs,
    axis=0,
)

native_ants_probs = meta_test[
    "p_ants_native_mean"
].to_numpy()

simple_average = np.mean(
    np.column_stack([
        eda6_mean,
        ants_similarity_mean,
        profile_test_probs,
        native_ants_probs,
    ]),
    axis=1,
)

comparison = {
    "EDA6 ensemble": eda6_mean,
    "ANTs na similarity": ants_similarity_mean,
    "Profile MLP": profile_test_probs,
    "Natywny ANTs": native_ants_probs,
    "Prosta średnia 4 metod": simple_average,
    "Meta-model": final_test_probs,
}

results = []

for name, probabilities in comparison.items():
    probabilities = np.clip(
        probabilities,
        0.001,
        0.999,
    )

    results.append({
        "model": name,
        "log_loss": log_loss(
            y_meta_test,
            probabilities,
        ),
        "AUROC": roc_auc_score(
            y_meta_test,
            probabilities,
        ),
    })

comparison_df = (
    pd.DataFrame(results)
    .sort_values("log_loss")
    .reset_index(drop=True)
)

display(comparison_df)

,model,log_loss,AUROC
0,EDA6 ensemble,0.320103,0.938591
1,Meta-model,0.333671,0.943415
2,ANTs na similarity,0.339847,0.933875
3,Prosta średnia 4 metod,0.405568,0.929593
4,Profile MLP,0.419651,0.896043
5,Natywny ANTs,1.515487,0.449648


In [45]:
from sklearn.base import clone

native_columns = {
    "p_ants_native_mean",
    "p_antspyx_main_mean",
}

feature_columns_no_native = [
    column
    for column in feature_columns
    if column not in native_columns
]

print("Cechy bez natywnego ANTs:")

for column in feature_columns_no_native:
    print(column)

Cechy bez natywnego ANTs:
p_eda6_fold_1
p_eda6_fold_2
p_eda6_fold_3
p_eda6_fold_4
p_eda6_fold_5
p_antspyx_fold_1
p_antspyx_fold_2
p_antspyx_fold_3
p_antspyx_fold_4
p_antspyx_fold_5
p_profiles


In [46]:
from pathlib import Path

import numpy as np
import pandas as pd


DATA_DIR = Path.cwd()

eda6_train = pd.read_csv(
    DATA_DIR / "meta_train_predictions_eda6.csv"
)

ants_similarity_train = pd.read_csv(
    DATA_DIR
    / "meta_train_predictions_antspyx_on_rotation_by_similarity.csv"
)

profiles_train = pd.read_csv(
    DATA_DIR / "meta_train_predictions_profiles.csv"
)

for frame in [
    eda6_train,
    ants_similarity_train,
    profiles_train,
]:
    frame["uid"] = frame["uid"].astype(str)
    frame["y_true"] = frame["y_true"].astype(int)

meta_train = (
    eda6_train
    .merge(
        ants_similarity_train,
        on=["uid", "y_true"],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        profiles_train,
        on=["uid", "y_true"],
        how="inner",
        validate="one_to_one",
    )
)

print("meta_train:", meta_train.shape)
display(meta_train.head())

meta_train: (1089, 13)


,uid,y_true,p_eda6_fold_1,p_eda6_fold_2,p_eda6_fold_3,p_eda6_fold_4,p_eda6_fold_5,p_antspyx_fold_1,p_antspyx_fold_2,p_antspyx_fold_3,p_antspyx_fold_4,p_antspyx_fold_5,p_profiles
0,xjn6fokz,0,0.153959,0.285420,0.213447,0.333463,0.266485,0.206324,0.044938,0.593849,0.170358,0.631159,0.388491
1,po6j0m3l,1,0.945385,0.831022,0.973613,0.934644,0.973652,0.594830,0.494980,0.708146,0.860590,0.795568,0.955722
2,45bg34iq,0,0.509882,0.632679,0.220572,0.405176,0.359730,0.457912,0.233320,0.771167,0.474506,0.576462,0.550863
3,g5rifwr0,1,0.997282,0.994022,0.994718,0.978875,0.982774,0.576289,0.739176,0.907781,0.928097,0.962708,0.302974
4,6e07fpy1,0,0.103359,0.166019,0.097250,0.091975,0.107103,0.061857,0.006627,0.183021,0.014706,0.087207,0.180184


In [47]:
train_features = {
    column
    for column in meta_train.columns
    if column.startswith("p_")
}

test_features = {
    column
    for column in meta_test.columns
    if column.startswith("p_")
}

print(
    "Tylko w treningu:",
    sorted(train_features - test_features),
)

print(
    "Tylko w teście:",
    sorted(test_features - train_features),
)

Tylko w treningu: []
Tylko w teście: ['p_ants_native_mean']


In [48]:
feature_columns_no_native = [
    column
    for column in meta_train.columns
    if (
        column.startswith("p_")
        and column in meta_test.columns
    )
]

assert len(feature_columns_no_native) == 11

X_meta_no_native = meta_train[
    feature_columns_no_native
].to_numpy(dtype=np.float32)

X_meta_test_no_native = meta_test[
    feature_columns_no_native
].to_numpy(dtype=np.float32)

y_meta = meta_train[
    "y_true"
].to_numpy(dtype=int)

y_meta_test = meta_test[
    "y_true"
].to_numpy(dtype=int)

print("Train:", X_meta_no_native.shape)
print("Test:", X_meta_test_no_native.shape)

Train: (1089, 11)
Test: (273, 11)


In [49]:
from sklearn.base import clone
import joblib

saved_meta = joblib.load(
    Path.cwd()
    / "2final_meta_classifier.joblib"
)

meta_model_template = saved_meta["model"]

meta_model_no_native = clone(
    meta_model_template
)

meta_model_no_native.fit(
    X_meta_no_native,
    y_meta,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [50]:
test_probs_no_native = (
    meta_model_no_native.predict_proba(
        X_meta_test_no_native
    )[:, 1]
)

test_probs_no_native = np.clip(
    test_probs_no_native,
    0.001,
    0.999,
)

test_classes_no_native = (
    test_probs_no_native >= 0.5
).astype(int)

print(
    "TEST log loss:",
    log_loss(
        y_meta_test,
        test_probs_no_native,
    ),
)

print(
    "TEST AUROC:",
    roc_auc_score(
        y_meta_test,
        test_probs_no_native,
    ),
)

print(
    "TEST accuracy:",
    accuracy_score(
        y_meta_test,
        test_classes_no_native,
    ),
)

print(
    "TEST F1:",
    f1_score(
        y_meta_test,
        test_classes_no_native,
    ),
)

print(
    confusion_matrix(
        y_meta_test,
        test_classes_no_native,
    )
)

TEST log loss: 0.3243447244167328
TEST AUROC: 0.9499186991869918
TEST accuracy: 0.8534798534798534
TEST F1: 0.8709677419354839
[[ 98  25]
 [ 15 135]]


In [51]:
NEW_META_MODEL_PATH = (
    Path.cwd()
    / "final_meta_classifier_reduced_profiles.joblib"
)

joblib.dump(
    {
        "model": meta_model_no_native,
        "feature_columns": (
            feature_columns_no_native
        ),
    },
    NEW_META_MODEL_PATH,
)

print(
    "Zapisano meta-model:",
    NEW_META_MODEL_PATH,
)

Zapisano meta-model: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons\src\notebooks\EDA6\final_meta_classifier_reduced_profiles.joblib


In [52]:
print(
    "Liczba cech modelu:",
    meta_model_no_native.n_features_in_,
)

print(
    "Kolumny:",
    feature_columns_no_native,
)

assert (
    meta_model_no_native.n_features_in_
    == len(feature_columns_no_native)
)

Liczba cech modelu: 11
Kolumny: ['p_eda6_fold_1', 'p_eda6_fold_2', 'p_eda6_fold_3', 'p_eda6_fold_4', 'p_eda6_fold_5', 'p_antspyx_fold_1', 'p_antspyx_fold_2', 'p_antspyx_fold_3', 'p_antspyx_fold_4', 'p_antspyx_fold_5', 'p_profiles']
